# CNN de referencia frente a YOLO26n-cls

En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Sube
`proyecto_renal.zip` a **Mi unidad** en Google Drive y ejecuta las celdas en orden. Colab descarga
Kaggle v2 y copia el código al disco de la sesión. Los resultados se escriben en Drive.

La prueba breve usa dos lotes de entrenamiento y valida únicamente sobre validación. Mantén
`EJECUTAR_COMPLETO = False` hasta que pasen ambos modelos. Después actívalo y ejecuta la última
celda. En sesiones nuevas vuelve a ejecutar las celdas de preparación. El programa omite corridas
completas y reinicia la corrida que se interrumpió. Los pesos y métricas de prueba se generan
cuando existen los diez entrenamientos.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
project_zip = drive_root / 'proyecto_renal.zip'
assert project_zip.is_file(), f'Sube {project_zip.name} a Mi unidad'
subprocess.run(['unzip', '-oq', str(project_zip), '-d', '/content'], check=True)
project = Path('/content/renal')
results = drive_root / 'resultados_renal_70_25_5'
assert (project / 'requirements.txt').is_file()
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=project, check=True)
# Comprobar ambos frameworks en procesos aislados para liberar la GPU antes de entrenar.
subprocess.run([sys.executable, '-c',
    "import tensorflow as tf; assert tf.config.list_physical_devices('GPU'), 'TensorFlow no detecta GPU'"],
    cwd=project, check=True)
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'PyTorch no detecta GPU'"],
    cwd=project, check=True)

In [ ]:
from hashlib import sha256

data = Path('/content/renal_kaggle_v2')
archive_hash = '7c4bad073d844775a57a235a3292570726b03603e4f0b8ed43f09e39cf7c09c3'
if not (data / 'raw/dataset/datos.csv').is_file():
    subprocess.run(['bash', 'data/download.sh', str(data)], cwd=project, check=True)
with (data / 'renal.zip').open('rb') as stream:
    from hashlib import file_digest
    assert file_digest(stream, 'sha256').hexdigest() == archive_hash
manifest = project / 'results/audit/manifest.csv'
assert sha256(manifest.read_bytes()).hexdigest() == 'c2a3b9888bae729b5f2c50289e4d2d8918fa15e699761a20178b7c3d1b77f8dd'
options = ['--root', str(data / 'raw'), '--manifest', str(manifest), '--output', str(results)]

In [ ]:
def run(phase, model, seed, smoke=False):
    command = [sys.executable, 'src/run.py', phase, model, str(seed), *options]
    if smoke:
        command.append('--smoke')
    subprocess.run(command, cwd=project, check=True)

for model in ('cnn', 'yolo'):
    run('train', model, 17, smoke=True)
    run('evaluate', model, 17, smoke=True)

In [ ]:
EJECUTAR_COMPLETO = False
if EJECUTAR_COMPLETO:
    for phase in ('train', 'evaluate'):
        for seed in (17, 27, 37, 47, 57):
            for model in ('cnn', 'yolo'):
                run(phase, model, seed)
    subprocess.run([sys.executable, 'src/aggregate.py', str(results), '--excel-fallback'], cwd=project, check=True)
else:
    print('Activa EJECUTAR_COMPLETO después de comprobar las dos pruebas breves.')